# Derived features and type-casting in pyspark

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, round, to_date, \
    year, month, dayofmonth, datediff, current_date, \
    upper, lower, trim, concat, regexp_replace, expr

In [2]:
spark = SparkSession.builder \
    .appName("BankingColumnOps") \
    .master("local[*]") \
    .getOrCreate()

In [3]:
df = spark.read.csv("../datasets/transactions_v2.csv", header=True, inferSchema=True)
df.printSchema()
df.show(truncate=False)

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- account_type: string (nullable = true)
 |-- balance: double (nullable = true)
 |-- transaction_amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- city: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- num_previous_defaults: integer (nullable = true)

+--------------+-----------+-----------+---+------------+--------+------------------+----------------+---------+----------------+------------+---------------------+
|transaction_id|customer_id|name       |age|account_type|balance |transaction_amount|transaction_type|city     |transaction_date|credit_score|num_previous_defaults|
+--------------+-----------+-----------+---+------------+--------+------------------+----------------+---------+----------------+------------

### Overwriting an existing column

In [4]:
df = df.withColumn('balance', col('balance').cast('double'))

### Adding constant columns with lit

In [5]:
from pyspark.sql.functions import lit 

In [6]:
df = df.withColumn("bank_name", lit('National Bank of India'))
df = df.withColumn('data_version', lit(2))
df = df.withColumn("is_processed", lit(True))

### Conditional columns

In [7]:
## Flagging high value transactions (say transaction > 10000 are high value)

df = df.withColumn('is_high_value',
                   when(col('transaction_amount')>10000, 'Yes')
                   .otherwise("No"))

In [8]:
df.select(col('is_high_value')).show()

+-------------+
|is_high_value|
+-------------+
|           No|
|          Yes|
|           No|
|           No|
|          Yes|
|           No|
|           No|
|           No|
|          Yes|
|           No|
+-------------+



### Chaining multiple conditions together

In [9]:
# Credit score risk banding 
df = df.withColumn("risk_band",
    when(col("credit_score") >= 750, "Excellent")
    .when(col("credit_score") >= 700, "Good")
    .when(col("credit_score") >= 650, "Fair")
    .when(col("credit_score") >= 600, "Poor")
    .otherwise("No Score")        
)

In [10]:
df.select(col('risk_band')).show()

+---------+
|risk_band|
+---------+
|     Good|
|     Fair|
| No Score|
|     Good|
| No Score|
|     Poor|
|     Fair|
|     Good|
| No Score|
|Excellent|
+---------+



### Filtering using multiple columns

In [11]:
# A transaction is "Suspicious" if it's a large debit from a low-balance account
df = df.withColumn("suspicious_flag",
    when(
        (col("transaction_type") == "Debit") &
        (col("transaction_amount") > 5000) &
        (col("balance") < 20000),
        "Suspicious"
    ).otherwise("Normal")
)

In [12]:
df.select(col('suspicious_flag')).show()

+---------------+
|suspicious_flag|
+---------------+
|         Normal|
|         Normal|
|         Normal|
|         Normal|
|         Normal|
|         Normal|
|         Normal|
|         Normal|
|         Normal|
|         Normal|
+---------------+



### Mathematical Derived columns

In [13]:
from pyspark.sql.functions import round, abs, sqrt, pow

- transaction-to-balance ratio

In [14]:
df = df.withColumn('txn_to_balance_ratio',
                   round(col('transaction_amount')/ col('balance'),4))

In [15]:
df.show()

{"ts": "2026-04-29 15:32:19.303", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[DIVIDE_BY_ZERO] Division by zero. Use `try_divide` to tolerate divisor being 0 and return NULL instead. If necessary set \"spark.sql.ansi.enabled\" to \"false\" to bypass this error. SQLSTATE: 22012", "context": {"file": "line 2 in cell [14]", "line": "", "fragment": "__truediv__", "errorClass": "DIVIDE_BY_ZERO"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o126.showString.\n: org.apache.spark.SparkArithmeticException: [DIVIDE_BY_ZERO] Division by zero. Use `try_divide` to tolerate divisor being 0 and return NULL instead. If necessary set \"spark.sql.ansi.enabled\" to \"false\" to bypass this error. SQLSTATE: 22012\n== DataFrame ==\n\"__truediv__\" was called from\nline 2 in cell [14]\n\r\n\tat org.apache.spark.sql.errors.QueryExecutionErrors$.divideByZeroError(QueryExecutionErrors.scala:205)\r\n\tat org.apache.spark.sql.errors.QueryExecutionErrors.d

ArithmeticException: [DIVIDE_BY_ZERO] Division by zero. Use `try_divide` to tolerate divisor being 0 and return NULL instead. If necessary set "spark.sql.ansi.enabled" to "false" to bypass this error. SQLSTATE: 22012
== DataFrame ==
"__truediv__" was called from
line 2 in cell [14]


- division by zero error comes, so we handle it 

In [16]:
df = df.withColumn("txn_to_balance_ratio",
    when(
        col("balance").isNull() | (col("balance") == 0),
        lit(None)
    ).otherwise(
        round(col("transaction_amount") / col("balance"), 4)
    )
)

In [17]:
df.show()

+--------------+-----------+-----------+---+------------+--------+------------------+----------------+---------+----------------+------------+---------------------+--------------------+------------+------------+-------------+---------+---------------+--------------------+
|transaction_id|customer_id|       name|age|account_type| balance|transaction_amount|transaction_type|     city|transaction_date|credit_score|num_previous_defaults|           bank_name|data_version|is_processed|is_high_value|risk_band|suspicious_flag|txn_to_balance_ratio|
+--------------+-----------+-----------+---+------------+--------+------------------+----------------+---------+----------------+------------+---------------------+--------------------+------------+------------+-------------+---------+---------------+--------------------+
|          T001|       C101|Arun Sharma| 34|     Savings| 45000.0|            5000.0|           Debit|   Mumbai|      2024-01-15|         720|                    0|National Bank of 

### Remaining balance after transaction

In [18]:
df = df.withColumn("balance_after_txn",
                   when(col('transaction_type')=='Debit',
                        col('balance') - col('transaction_amount'))
                   .when(col('transaction_type')=='Credit',
                         col('balance')+col('transaction_amount'))
                   .otherwise(col('balance')))

In [19]:
df.columns

['transaction_id',
 'customer_id',
 'name',
 'age',
 'account_type',
 'balance',
 'transaction_amount',
 'transaction_type',
 'city',
 'transaction_date',
 'credit_score',
 'num_previous_defaults',
 'bank_name',
 'data_version',
 'is_processed',
 'is_high_value',
 'risk_band',
 'suspicious_flag',
 'txn_to_balance_ratio',
 'balance_after_txn']

In [20]:
df.select('balance_after_txn').show()

+-----------------+
|balance_after_txn|
+-----------------+
|          40000.0|
|         135000.0|
|           6000.0|
|          37000.0|
|          25000.0|
|          14000.0|
|         127000.0|
|          66500.0|
|             NULL|
|          98000.0|
+-----------------+



### Casting to another datatype

In [21]:
df = df.withColumn('credit_score', col('credit_score').cast('integer'))

df = df.withColumn('balance', col('balance').cast('double')) 


### Date column operations

In [22]:
df.select('transaction_date').show()

+----------------+
|transaction_date|
+----------------+
|      2024-01-15|
|      2024-01-16|
|      2024-01-17|
|      2024-01-18|
|      2024-01-19|
|      2024-01-20|
|      2024-01-21|
|      2024-01-22|
|      2024-01-23|
|      2024-01-24|
+----------------+



In [23]:
df = df.withColumn('transaction_date',
                   to_date(col('transaction_date'), 'yyyy-MM-dd'))

In [24]:
# Extract year, month, day from the date
df = df.withColumn("txn_year",  year(col("transaction_date")))
df = df.withColumn("txn_month", month(col("transaction_date")))
df = df.withColumn("txn_day",   dayofmonth(col("transaction_date")))

In [25]:
df.select(['transaction_date','txn_year','txn_month','txn_day']).show()

+----------------+--------+---------+-------+
|transaction_date|txn_year|txn_month|txn_day|
+----------------+--------+---------+-------+
|      2024-01-15|    2024|        1|     15|
|      2024-01-16|    2024|        1|     16|
|      2024-01-17|    2024|        1|     17|
|      2024-01-18|    2024|        1|     18|
|      2024-01-19|    2024|        1|     19|
|      2024-01-20|    2024|        1|     20|
|      2024-01-21|    2024|        1|     21|
|      2024-01-22|    2024|        1|     22|
|      2024-01-23|    2024|        1|     23|
|      2024-01-24|    2024|        1|     24|
+----------------+--------+---------+-------+



### Arithmetic operation on date columns

In [26]:
df = df.withColumn("days_since_txn",
                   datediff(current_date(), col('transaction_date')))

In [27]:
df.select('days_since_txn').show()

+--------------+
|days_since_txn|
+--------------+
|           835|
|           834|
|           833|
|           832|
|           831|
|           830|
|           829|
|           828|
|           827|
|           826|
+--------------+



In [28]:
from pyspark.sql.functions import date_add, date_sub, months_between

In [29]:
# Add 30 days to transaction date (e.g., payment due date)
df = df.withColumn("payment_due_date",
    date_add(col("transaction_date"), 30)
)

# Subtract 7 days (lookback window)
df = df.withColumn("lookback_start",
    date_sub(col("transaction_date"), 7)
)

In [30]:
# Months between two dates (loan tenure calculation)
df = df.withColumn("months_active",
    months_between(current_date(), col("transaction_date"))
)

### String operations

In [31]:
from pyspark.sql.functions import upper, lower, trim, length, \
                                   concat, concat_ws, substring, \
                                   regexp_replace, split

In [32]:
# Standardizing name to uppercase 
df = df.withColumn("name_upper", upper(col("name")))

# Lowercase city for joining with reference tables
df = df.withColumn("city_lower", lower(col("city")))

# Removing leading/trailing spaces
df = df.withColumn("name", trim(col("name")))

# Length of name (useful for validation)
df = df.withColumn("name_length", length(col("name")))

In [34]:
# Concatenating columns 
df = df.withColumn("customer_label",
    concat(col("customer_id"), lit(" - "), col("name"))
)

df.select(col('customer_label')).show()

+------------------+
|    customer_label|
+------------------+
|C101 - Arun Sharma|
|  C102 - Priya Sen|
|  C103 - Rahul Das|
|C101 - Arun Sharma|
| C104 - Meena Iyer|
| C105 - Suresh Roy|
|  C102 - Priya Sen|
|C106 - Fatima Khan|
|  C103 - Rahul Das|
|C107 - Vikram Nair|
+------------------+



In [36]:

# concat_ws (with separator)
df = df.withColumn("customer_label",
    concat_ws(" | ", col("customer_id"), col("name"), col("city"))
)
df.select(col('customer_label')).show(truncate=False)

+------------------------------+
|customer_label                |
+------------------------------+
|C101 | Arun Sharma | Mumbai   |
|C102 | Priya Sen | Delhi      |
|C103 | Rahul Das | Kolkata    |
|C101 | Arun Sharma | Mumbai   |
|C104 | Meena Iyer | Chennai   |
|C105 | Suresh Roy | Pune      |
|C102 | Priya Sen | Delhi      |
|C106 | Fatima Khan | Hyderabad|
|C103 | Rahul Das | Kolkata    |
|C107 | Vikram Nair | Bangalore|
+------------------------------+



In [38]:
# Extracting first 4 characters of transaction_id (prefix code)
df = df.withColumn("txn_prefix",
    substring(col("transaction_id"), 1, 3)
)
# substring(col, start_position, length)
# Position starts at 1, not 0 — unlike Python

df.select([col('transaction_id'),col('txn_prefix')]).show()

+--------------+----------+
|transaction_id|txn_prefix|
+--------------+----------+
|          T001|       T00|
|          T002|       T00|
|          T003|       T00|
|          T004|       T00|
|          T005|       T00|
|          T006|       T00|
|          T007|       T00|
|          T008|       T00|
|          T009|       T00|
|          T010|       T01|
+--------------+----------+



### Renaming a column

In [40]:
df = df.withColumnRenamed("transaction_amount", "txn_amount")


- renaming many columns at once

In [41]:
# Rename ALL columns at once by passing a new list
new_column_names = [
    "txn_id", "cust_id", "cust_name", "cust_age",
    "acct_type", "acct_balance", "txn_amount",
    "txn_type", "cust_city", "txn_date",
    "credit_score", "prev_defaults"
]

df_renamed = df.toDF(*new_column_names)

AnalysisException: [ASSIGNMENT_ARITY_MISMATCH] The number of columns or variables assigned or aliased: 32 does not match the number of source expressions: 12. SQLSTATE: 42802

`the list size should be same as the no of columns in the df, here it is different hence throwing error`

In [43]:
df_clean = df.select(
    col("transaction_id").alias("txn_id"),
    col("customer_id").alias("cust_id"),
    col("name").alias("customer_name"),
    col("balance").alias("account_balance"),
    col("transaction_type").alias("txn_type"),
    col("transaction_date").alias("txn_date"),
    col("credit_score")     # keep as-is 
)

In [44]:
df_clean.columns

['txn_id',
 'cust_id',
 'customer_name',
 'account_balance',
 'txn_type',
 'txn_date',
 'credit_score']